# 13 - Train Multi-Head n-step DQN Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but trains **several Q heads**, each with its own n-step backup, instead of one one-step head:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and **one `RegressionHead` per horizon**.
4. Call `DqnObjective` once per head (`nstep_gate(n=...)`) and **sum** the losses. Save with `push_model_to_hub`.

An n-step target is the discounted sum of the next `n` rewards plus a delayed state-value bootstrap at `s_{t+n}` (`max Q` when `temperature=0`, or `α logsumexp(Q / α)` when `temperature=α > 0`):

`G_t^{(n)} = r_{t+1} + γ_{t+1} r_{t+2} + … + (∏ γ) V(s_{t+n})`

`n=1` is ordinary one-step DQN. A longer `n` puts more of the return on the observed rewards and less on the delayed network. If the window hits a run break (`sequence_id` / `grouping_field`) or the end of the batch, the return truncates and bootstraps at the last in-run next state. Episode / task done-code gammas multiply later rewards and the bootstrap.

One `DqnObjective` trains one Q tensor. `gate=nstep_gate(n=...)` is the n-step return (`n=1` is ordinary one-step DQN). This notebook builds three heads — `action_value_1`, `action_value_3`, `action_value_5` — and calls the objective three times, passing that head's online and delayed tensors. The heads share the backbone; `get_action` reads the longest-horizon head (`action_source=head_keys[-1]`). The delayed model copies every Q head and shares the backbone.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount, nstep_gate
from mouse_core.models import Model, Polyak, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-nstep-offline"          # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-nstep-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by each head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
N_STEPS = (1, 3, 5)                           # backup horizon of each Q head (one DqnObjective per n)
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "skip": 0.0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "skip": 0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
    group_prefix="action,observation,r=reward,d=done\n",
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves. `use_norm` (required, saved with the model) keeps (`True`) or drops (`False`) the final RMSNorm.
- One `RegressionHead` per horizon. Each predicts one value per discrete action from the same pooled features. `use_norm` (required) keeps (`True`) or drops (`False`) the head's input RMSNorm.

The backbone exposes `hidden_dim`, and the embedder and heads use that same value so the pieces connect cleanly.

`Tokenizer` text fields match `02`: comma-separated action / observation, `r=` / `d=` when nonzero, and a const newline readout flagged `head_output: True`.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. Heads are a dict keyed by `action_value_{n}` for each `n` in `N_STEPS`. `action_source` is the longest-horizon head name so `get_action` / `15_inference.ipynb` read that Q.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)



def make_q_head() -> RegressionHead:
    return RegressionHead(
        in_features=backbone.hidden_dim,
        out_features=MAX_ACTIONS,
        hidden_dim=backbone.hidden_dim,
        num_layers=1,
        scale=0.1,
        use_norm=True,
    )


head_keys = tuple(f"action_value_{n}" for n in N_STEPS)
heads = {key: make_q_head() for key in head_keys}

model = Model(
    backbone=backbone,
    heads=heads,
    action_source=head_keys[-1],
    reasoner=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step predictions for **every** Q head.
3. Each `DqnObjective` is called on the same `objective_data` and the Q tensors for that head. The training loss is the **sum** of those scalars.
4. `AdamW` updates weights. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q comes from the delayed model: `delayed_model = model.copy(heads=True, backbone=False, reasoner=False)` copies every Q head and keeps the online backbone (including token embeddings). After the online forward, `delayed_model.head(h=delayed_model.pool(output=out))` runs only those heads on the online hidden states under `torch.no_grad()` — no second backbone pass. `polyak.update(tau_heads=POLYAK_TAU_HEADS)` interpolates the copied heads toward the online model after the optimizer step (`0` frozen, `1` copy of the online heads). `tau_backbone` is omitted because the backbone was not copied. Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`gate=nstep_gate(n=...)` is the n-step return. Along a run the target of horizon `n` is

`G_i^{(n)} = r_i + γ_i r_{i+1} + … + (∏_{k=0}^{n-1} γ_{i+k}) V(s_{i+n})`,

where `V` is the delayed state value of **that** head: `max Q` when `temperature=0`, or the SAC soft value `α logsumexp(Q / α)` when `temperature=α > 0` (same required knob as `get_action`). `n=1` is `r + γ V`. A window that cannot look `n` steps ahead (run break or end of the sampled sequence) bootstraps at the last in-run next state.

`reward` is called with the unpacked `objective_data` columns. `boundary_reward` is the standard lookup: `(scale × episode scale × task scale) * reward + shift + episode shift + task shift`. Scale extras are `1.0` and shift extras are `0.0` when the matching code is `0`. `reward=None` / `value=None` / `discount=None` skip that callable. When a task ends both extras fire. `value` is called as `value(value=..., **objective_data)`. `boundary_value` is the same lookup on online and delayed Q. `discount` is called with the unpacked `objective_data` columns. The gamma at each lookahead step multiplies later rewards and the bootstrap, so a `0` gamma ends the sum and a non-zero truncation gamma carries it, discounted. The window never reads across a run break (`sequence_id` / `grouping_field`).


In [ ]:
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.copy(heads=True, backbone=False, reasoner=False)
polyak = Polyak(online=model, delayed=delayed_model)
objectives = [
    DqnObjective(
        gate=nstep_gate(n=n),
        double=False,
        reward=None,
        value=None,
        discount=boundary_discount(
            gamma_step=1.0,
            gamma_episode_terminal=1.0,
            gamma_episode_truncated=1.0,
            gamma_task_terminal=0.0,
            gamma_task_truncated=0.0,
        ),
        grouping_field="task_index",
        temperature=0.0,
    )
    for n, key in zip(N_STEPS, head_keys)
]

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objectives: list[DqnObjective], loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_predictions = delayed_model.head(h=delayed_model.pool(output=out))
        batch = to_device(data=objective_data, device=device)
        loss = None
        metrics = {}
        for key, objective in zip(head_keys, objectives):
            part, part_metrics = objective(
                objective_data=batch,
                predictions=out.predictions[key],
                delayed_predictions=delayed_predictions[key],
            )
            loss = part if loss is None else loss + part
            metrics.update(
                {f"{key}_{name}": value for name, value in part_metrics.items()}
            )
        assert loss is not None
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(tau_heads=POLYAK_TAU_HEADS)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb` (`get_action` reads `action_value_5`).


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objectives=objectives,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    parts = "  ".join(
        f"n{n}={metrics[f'action_value_{n}_action_value']:.4f}" for n in N_STEPS
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  {parts}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")